In [1]:
import json_tricks
answer = {}


inputs = json_tricks.load(open('inputs/inputs.json', 'r'))

# Backpropagation Practice

You are given a graph of operations:

<img src="imgs/task_2_4_graph_02.png" width="400" height="300" />

# Task 1

Write a function that calculates the value of this graph for any given input vector `x` and set of coefficients $b_1, b_2, c_1, c_2$ packed as vector of weights `w`.
In this function also return all the $z$ values.

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def graph_value(x, w):
    z = []
    z.append(x[0] + w[0]) 
    z.append(x[1] + w[1])

    z.append(sigmoid(z[0]))
    z.append(sigmoid(z[1]))

    z.append(np.tanh(z[1]))
    z.append(z[4] * w[3])

    z.append(z[0] * z[3])
    z.append(z[6] * w[2])

    z.append(z[2] * z[5])
    y = z[7] + z[8]
    return y, np.array(z)

In [14]:
answer['graph_value'] = [graph_value(**input) for input in inputs]

# Task 2

In this graph, find all direct derivatives of all the values over $z$ or $x$ or $w$.

For example, if $y = z_8 + z_9$, you need to find only $\partial_{z_9} y$ and $\partial_{z_8} y$ for that operation.

You have to do that for all the intermediate signals.

Write the result in the form of a dictionary, for example, $\partial_{z_9} y$:

```
['dy_dz9'] = 1
```

In [1]:
def graph_derivatives(x, w):
    y, z = graph_value(x, w)
    res = {}

    res['dy_dz9'] = 1
    res['dy_dz8'] = 1

    res['dz9_dz3'] = z[5]
    res['dz9_dz6'] = z[2]

    res['dz8_dz7'] = w[2]
    res['dz8_dc1'] = z[6]

    res['dz7_dz1'] = z[3]
    res['dz7_dz4'] = z[0]

    res['dz6_dz5'] = w[3]
    res['dz6_dc2'] = z[4]

    res['dz5_dz2'] = 1 - z[4] ** 2

    res['dz4_dz2'] = z[3] * (1 - z[3])

    res['dz3_dz1'] = z[2] * (1 - z[2])

    res['dz2_dx2'] = 1
    res['dz2_db2'] = 1

    res['dz1_dx1'] = 1
    res['dz1_db1'] = 1
        
    return res

In [17]:
answer['graph_derivatives'] = [graph_derivatives(**input) for input in inputs]

# Task 3

Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_1} y$
- calculate that derivative

In [ ]:
def dy_dc1(x, w):
    selected_derivs = {}
    derivs = graph_derivatives(x, w)
    
    selected_derivs['dy_dz8'] = derivs['dy_dz8']
    selected_derivs['dz8_dc1'] = derivs['dz8_dc1']

    dy_dc1 = selected_derivs['dy_dz8'] * selected_derivs['dz8_dc1']

    return selected_derivs, dy_dc1

In [19]:
answer['dy_dc1'] = [dy_dc1(**input) for input in inputs]

# Task 4
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{c_2} y$
- calculate that derivative

In [ ]:
def dy_dc2(x, w):
    selected_derivs = {}

    derivs = graph_derivatives(x, w)

    selected_derivs['dy_dz9'] = derivs['dy_dz9']
    selected_derivs['dz9_dz6'] = derivs['dz9_dz6']
    selected_derivs['dz6_dc2'] = derivs['dz6_dc2']

    dy_dc2 = selected_derivs['dy_dz9'] * selected_derivs['dz9_dz6'] * selected_derivs['dz6_dc2']

    return selected_derivs, dy_dc2

In [21]:
answer['dy_dc2'] = [dy_dc2(**input) for input in inputs]

# Task 5
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_1} y$
- calculate that derivative

In [ ]:
def dy_db1(x, w):
    selected_derivs = {}
    dy_db1 = 0

    derivs = graph_derivatives(x, w)

    selected_derivs['dy_dz9'] = derivs['dy_dz9']
    selected_derivs['dz9_dz3'] = derivs['dz9_dz3']
    selected_derivs['dz3_dz1'] = derivs['dz3_dz1']
    selected_derivs['dz1_db1'] = derivs['dz1_db1']

    selected_derivs['dy_dz8'] = derivs['dy_dz8']
    selected_derivs['dz8_dz7'] = derivs['dz8_dz7']
    selected_derivs['dz7_dz1'] = derivs['dz7_dz1']

    left = selected_derivs['dy_dz9'] * selected_derivs['dz9_dz3'] * selected_derivs['dz3_dz1'] * selected_derivs['dz1_db1']
    right = selected_derivs['dy_dz8'] * selected_derivs['dz8_dz7'] * selected_derivs['dz7_dz1'] * selected_derivs['dz1_db1']
    dy_db1 = left + right
    
    return selected_derivs, dy_db1

In [23]:
answer['dy_db1'] = [dy_db1(**input) for input in inputs]

# Task 6
Using the values of the derivatives, calculated above:
- extract a dictionary of values that are needed to calculate $\partial_{b_2} y$
- calculate that derivative

In [ ]:
def dy_db2(x, w):
    selected_derivs = {}
    dy_db2 = 0

    derivs = graph_derivatives(x, w)

    selected_derivs['dy_dz8'] = derivs['dy_dz8']
    selected_derivs['dz8_dz7'] = derivs['dz8_dz7']
    selected_derivs['dz7_dz4'] = derivs['dz7_dz4']
    selected_derivs['dz4_dz2'] = derivs['dz4_dz2']
    selected_derivs['dz2_db2'] = derivs['dz2_db2']

    selected_derivs['dy_dz9'] = derivs['dy_dz9']
    selected_derivs['dz9_dz6'] = derivs['dz9_dz6']
    selected_derivs['dz6_dz5'] = derivs['dz6_dz5']
    selected_derivs['dz5_dz2'] = derivs['dz5_dz2']

    left = selected_derivs['dy_dz8'] * selected_derivs['dz8_dz7'] * selected_derivs['dz7_dz4'] * selected_derivs['dz4_dz2'] * selected_derivs['dz2_db2']
    right = selected_derivs['dy_dz9'] * selected_derivs['dz9_dz6'] * selected_derivs['dz6_dz5'] * selected_derivs['dz5_dz2'] * selected_derivs['dz2_db2']
    
    dy_db2 = left + right

    return selected_derivs, dy_db2

In [25]:
answer['dy_db2'] = [dy_db2(**input) for input in inputs]

In [ ]:
json_tricks.dump(answer, '.answer.json')